# Training Mask R-CNN (Detectron2) on SUN RGB-D

The same architecture (Mask R-CNN, ResNet-50 + FPN), but implemented in **Detectron2** —
the reference library by the authors of Mask R-CNN (FAIR). Comparing the two
implementations on identical subsets provides extra material for chapter 2: how
much the result depends on implementation details and the training recipe rather
than the architecture alone.

In [ ]:
from pathlib import Path
import json 
import os

import random, cv2
import matplotlib.pyplot as plt
import pandas as pd

from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data.datasets import register_coco_instances
from detectron2.data import MetadataCatalog, DatasetCatalog, build_detection_test_loader
from detectron2.engine import DefaultTrainer, DefaultPredictor, hooks
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.checkpoint import DetectionCheckpointer

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
CONFIG = "std10"          # "std10" (10 classes) | "min400" (20) | "min100" (41)
SPLIT_SUBDIR = {"std10": "splits", "min400": "splits_min400",
                "min100": "splits_min100"}[CONFIG]

DATA_ZIP  = Path("/content/drive/MyDrive/sunrgbd_data.zip")
DATA_ROOT = Path("/content/sunrgbd")
OUT_DIR   = Path(f"/content/drive/MyDrive/vkr_runs/maskrcnn_detectron2_{CONFIG}")
SPLIT_DIR = DATA_ROOT / SPLIT_SUBDIR

# unpack once per Colab session (the zip is stored uncompressed, ~5 min from Drive)
if not SPLIT_DIR.is_dir():
    assert DATA_ZIP.is_file(), f"archive not found on Drive: {DATA_ZIP}"
    print("Unpacking archive...")
    !unzip -q -n "{DATA_ZIP}" -d "{DATA_ROOT}"

OUT_DIR.mkdir(parents=True, exist_ok=True)
for s in ("train", "val", "test"):
    assert (SPLIT_DIR / f"instances_{s}.json").is_file(), f"missing instances_{s}.json"
print("Config:", CONFIG)
print("Data:", DATA_ROOT)
print("Outputs:", OUT_DIR)

In [ ]:
SEED         = 42
BATCH_SIZE   = 4
NUM_WORKERS  = 2
EPOCHS       = 12
BASE_LR      = 0.005
LR_DECAY_EPOCHS = [8, 11]     # epochs at which LR ÷ 10
WARMUP_ITERS = 500

_train = json.loads((SPLIT_DIR / "instances_train.json").read_text())
NUM_CLASSES = len(_train["categories"])   # no background class in Detectron2
n_train = len(_train["images"])
ITERS_PER_EPOCH = (n_train + BATCH_SIZE - 1) // BATCH_SIZE
MAX_ITER = EPOCHS * ITERS_PER_EPOCH
STEPS = tuple(e * ITERS_PER_EPOCH for e in LR_DECAY_EPOCHS)
print(f"{NUM_CLASSES} classes; train: {n_train} images, "
      f"{ITERS_PER_EPOCH} iterations/epoch, {MAX_ITER} iterations total, "
      f"LR decay at {STEPS}")

---
## Dataset

Detectron2 works with a dataset registry. `register_coco_instances` reads our
COCO files; the image paths inside are relative, so the `DATA_ROOT` root is
passed in. Category ids (1–10) are mapped by the library itself to contiguous
indices 0–9.

In [ ]:
for s in ("train", "val", "test"):
    name = f"sunrgbd_{s}"
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name); MetadataCatalog.remove(name)
    register_coco_instances(name, {}, str(SPLIT_DIR / f"instances_{s}.json"), str(DATA_ROOT))

# loading the dicts also fills in metadata (thing_classes) lazily
for s in ("train", "val", "test"):
    print(f"{s:5s}: {len(DatasetCatalog.get(f'sunrgbd_{s}'))} images")

meta = MetadataCatalog.get("sunrgbd_train")
print("Classes:", meta.thing_classes)

In [ ]:
# visual check of the annotations using Detectron2 tooling

rng = random.Random(SEED)
samples = rng.sample(DatasetCatalog.get("sunrgbd_train"), 3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, d in zip(axes, samples):
    img = cv2.imread(d["file_name"])[:, :, ::-1]
    vis = Visualizer(img, metadata=meta, scale=0.8)
    out = vis.draw_dataset_dict(d)
    ax.imshow(out.get_image()); ax.axis("off")
plt.tight_layout(); plt.show()

---
## Configuration

The base is `mask_rcnn_R_50_FPN_3x.yaml` from the Model Zoo: Mask R-CNN,
ResNet-50 + FPN, weights pre-trained on COCO (the "3x" schedule). Only the
number of classes, the batch size, and the training schedule are changed.

In [ ]:
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

cfg.DATASETS.TRAIN = ("sunrgbd_train",)
cfg.DATASETS.TEST  = ("sunrgbd_val",)          # in-training evaluation — on the validation set
cfg.DATALOADER.NUM_WORKERS = NUM_WORKERS

cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
cfg.SOLVER.BASE_LR = BASE_LR
cfg.SOLVER.MAX_ITER = MAX_ITER
cfg.SOLVER.STEPS = STEPS
cfg.SOLVER.WARMUP_ITERS = WARMUP_ITERS
cfg.SOLVER.CHECKPOINT_PERIOD = ITERS_PER_EPOCH      # checkpoint every epoch
cfg.TEST.EVAL_PERIOD = ITERS_PER_EPOCH              # evaluation every epoch
cfg.SEED = SEED
cfg.OUTPUT_DIR = str(OUT_DIR)

print(f"MAX_ITER={cfg.SOLVER.MAX_ITER}, STEPS={cfg.SOLVER.STEPS}, "
      f"eval every {cfg.TEST.EVAL_PERIOD} iterations")

---
## Training

`DefaultTrainer` + two additions:

* **COCOEvaluator on the validation set** every epoch — the same metrics
  (mask AP) as in notebook 02;
* **BestCheckpointer** — saves `model_best.pth` by the highest `segm/AP`.

`resume_or_load(resume=True)` allows training to continue from the last
checkpoint after a Colab session drop — just rerun the cells from the top.

In [ ]:
class Trainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "eval_val")
        return COCOEvaluator(dataset_name, output_dir=output_folder)


trainer = Trainer(cfg)
trainer.register_hooks([hooks.BestCheckpointer(
    cfg.TEST.EVAL_PERIOD, trainer.checkpointer, "segm/AP", mode="max")])
trainer.resume_or_load(resume=True)
trainer.train()

In [ ]:
# training curves from metrics.json 
lines = []
with open(OUT_DIR / "metrics.json") as f:
    for line in f:
        try:
            lines.append(json.loads(line))
        except json.JSONDecodeError:
            continue
m = pd.DataFrame(lines)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

loss = m.dropna(subset=["total_loss"])
axes[0].plot(loss.iteration, loss.total_loss, lw=0.7, color="#2f6f9f")
axes[0].set_xlabel("Iteration"); axes[0].set_title("Total loss"); axes[0].grid(alpha=0.3)

for col, lbl in [("loss_cls", "cls"), ("loss_box_reg", "box"),
                 ("loss_mask", "mask"), ("loss_rpn_cls", "rpn_cls"),
                 ("loss_rpn_loc", "rpn_loc")]:
    if col in m:
        sub = m.dropna(subset=[col])
        axes[1].plot(sub.iteration, sub[col], lw=0.7, label=lbl)
axes[1].set_xlabel("Iteration"); axes[1].set_title("Loss components")
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3)

if "segm/AP" in m:
    ap = m.dropna(subset=["segm/AP"])
    axes[2].plot(ap.iteration, ap["segm/AP"], "-o", ms=3, color="#c0392b", label="mask AP")
    if "bbox/AP" in m:
        ap2 = m.dropna(subset=["bbox/AP"])
        axes[2].plot(ap2.iteration, ap2["bbox/AP"], "-o", ms=3, color="#2f6f9f", label="box AP")
    axes[2].set_xlabel("Iteration"); axes[2].set_title("Validation quality (val)")
    axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Final evaluation on the test set

Evaluates the model that is **best by validation** (`model_best.pth`). The result
is the full set of COCO metrics (including per-class AP).

In [ ]:
best = OUT_DIR / "model_best.pth"
weights = best if best.is_file() else OUT_DIR / "model_final.pth"
print("Weights:", weights.name)

model = Trainer.build_model(cfg)
DetectionCheckpointer(model, save_dir=cfg.OUTPUT_DIR).load(str(weights))

evaluator = COCOEvaluator("sunrgbd_test", output_dir=str(OUT_DIR / "eval_test"))
loader = build_detection_test_loader(cfg, "sunrgbd_test")
results = inference_on_dataset(model, loader, evaluator)

segm = results["segm"]
overall = {k: v for k, v in segm.items() if not k.startswith("AP-")}
per_class = {k[3:]: v for k, v in segm.items() if k.startswith("AP-")}

overall_df = pd.DataFrame([overall]).T.rename(columns={0: "value"}).round(2)
overall_df.to_csv(OUT_DIR / "test_metrics.csv")
print("\nFinal segm metrics (test):"); display(overall_df)

per_class_df = (pd.Series(per_class, name="mask AP").sort_values(ascending=False)
                .to_frame().round(2))
per_class_df.to_csv(OUT_DIR / "test_per_class_ap.csv")
print("Per-class mask AP:"); display(per_class_df)

In [ ]:
# extra test metrics from the saved predictions (no GPU needed):
# per-class precision / recall / F1, confusion matrix, mean IoU of matched masks
# COCOEvaluator dumps raw predictions to eval_test/coco_instances_results.json
import numpy as np
from pycocotools.coco import COCO
from pycocotools import mask as mask_utils

CONF_THRESH = 0.5   # confidence threshold (P/R at fixed conf, like YOLO reports)
IOU_THRESH  = 0.5   # mask IoU threshold for matching predictions to GT

coco_gt = COCO(str(SPLIT_DIR / "instances_test.json"))
PRED_FILE = OUT_DIR / "eval_test" / "coco_instances_results.json"
preds = [p for p in json.loads(PRED_FILE.read_text())
         if p["score"] >= CONF_THRESH and "segmentation" in p]

cat_ids   = sorted(coco_gt.getCatIds())
cat_names = [coco_gt.loadCats(c)[0]["name"] for c in cat_ids]
cat_idx   = {c: i for i, c in enumerate(cat_ids)}
K = len(cat_ids)

# confusion matrix: rows = GT classes (+ background), cols = predicted (+ missed)
conf = np.zeros((K + 1, K + 1), dtype=int)
tp, fp, fn = np.zeros(K, int), np.zeros(K, int), np.zeros(K, int)
matched_ious = []

preds_by_img = {}
for p in preds:
    preds_by_img.setdefault(p["image_id"], []).append(p)

for img_id in coco_gt.getImgIds():
    gts = coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=img_id, iscrowd=False))
    dts = sorted(preds_by_img.get(img_id, []), key=lambda p: -p["score"])
    gt_rles = [coco_gt.annToRLE(a) for a in gts]
    dt_rles = [d["segmentation"] for d in dts]
    ious = (mask_utils.iou(dt_rles, gt_rles, [0] * len(gt_rles))
            if gt_rles and dt_rles else np.zeros((len(dt_rles), len(gt_rles))))

    # greedy class-agnostic matching by descending score
    gt_used = set()
    for di, d in enumerate(dts):
        best_gi, best_iou = -1, IOU_THRESH
        for gi in range(len(gts)):
            if gi not in gt_used and ious[di, gi] >= best_iou:
                best_gi, best_iou = gi, ious[di, gi]
        dk = cat_idx[d["category_id"]]
        if best_gi >= 0:
            gt_used.add(best_gi)
            gk = cat_idx[gts[best_gi]["category_id"]]
            conf[gk, dk] += 1
            matched_ious.append(best_iou)
            if gk == dk:
                tp[dk] += 1
            else:                     # localized but misclassified
                fp[dk] += 1; fn[gk] += 1
        else:                         # no GT match -> false positive on background
            conf[K, dk] += 1
            fp[dk] += 1
    for gi, g in enumerate(gts):      # unmatched GT -> missed object
        if gi not in gt_used:
            gk = cat_idx[g["category_id"]]
            conf[gk, K] += 1
            fn[gk] += 1

prec = tp / np.maximum(tp + fp, 1)
rec  = tp / np.maximum(tp + fn, 1)
f1   = 2 * prec * rec / np.maximum(prec + rec, 1e-9)

table = pd.DataFrame({"precision": prec, "recall": rec, "F1": f1,
                      "TP": tp, "FP": fp, "FN": fn}, index=cat_names)
mp = tp.sum() / max(tp.sum() + fp.sum(), 1)
mr = tp.sum() / max(tp.sum() + fn.sum(), 1)
table.loc["micro avg"] = [mp, mr, 2 * mp * mr / max(mp + mr, 1e-9),
                          tp.sum(), fp.sum(), fn.sum()]
table.loc["macro avg"] = [prec.mean(), rec.mean(), f1.mean(),
                          tp.sum(), fp.sum(), fn.sum()]
table = table.round(4)
table.to_csv(OUT_DIR / "test_pr_f1.csv")
print(f"conf >= {CONF_THRESH}, mask IoU >= {IOU_THRESH}")
print(f"Mean IoU of matched masks: {np.mean(matched_ious):.4f}")
display(table)

# confusion matrix normalized by GT row
norm = conf / np.maximum(conf.sum(axis=1, keepdims=True), 1)
row_lbl = cat_names + ["background"]
col_lbl = cat_names + ["missed"]
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(K + 1)); ax.set_xticklabels(col_lbl, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(K + 1)); ax.set_yticklabels(row_lbl, fontsize=8)
for i in range(K + 1):
    for j in range(K + 1):
        if conf[i, j]:
            ax.text(j, i, conf[i, j], ha="center", va="center", fontsize=7,
                    color="white" if norm[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title(f"Confusion matrix (mask IoU >= {IOU_THRESH}, conf >= {CONF_THRESH})")
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# qualitative examples on test images

cfg.MODEL.WEIGHTS = str(weights)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
predictor = DefaultPredictor(cfg)

test_dicts = rng.sample(DatasetCatalog.get("sunrgbd_test"), 4)
fig, axes = plt.subplots(len(test_dicts), 2, figsize=(12, 5 * len(test_dicts)))
for row, d in enumerate(test_dicts):
    img = cv2.imread(d["file_name"])[:, :, ::-1]

    vis_gt = Visualizer(img, metadata=meta, scale=0.8).draw_dataset_dict(d)
    axes[row, 0].imshow(vis_gt.get_image()); axes[row, 0].axis("off")
    axes[row, 0].set_title("Ground truth", fontsize=9)

    out = predictor(img[:, :, ::-1])
    vis_pr = Visualizer(img, metadata=meta, scale=0.8).draw_instance_predictions(
        out["instances"].to("cpu"))
    axes[row, 1].imshow(vis_pr.get_image()); axes[row, 1].axis("off")
    axes[row, 1].set_title("Mask R-CNN / Detectron2 (score>0.5)", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "qualitative_test.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Inference timing

The protocol is identical across notebooks 02/03/04 (section 2.3 compares the
numbers directly): end-to-end single-image latency from a decoded image in
memory — preprocessing + forward pass + postprocessing, batch = 1,
10 warmup runs, 100 timed runs on test images, GPU-synchronized timing.

In [ ]:
import time
import numpy as np
import torch

N_WARMUP, N_RUNS = 10, 100
cfg.MODEL.WEIGHTS = str(weights)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
predictor = DefaultPredictor(cfg)

dicts = DatasetCatalog.get("sunrgbd_test")[:N_WARMUP + N_RUNS]
arrs = [cv2.imread(d["file_name"]) for d in dicts]

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

for a in arrs[:N_WARMUP]:
    predictor(a)
_sync()
times = []
for a in arrs[N_WARMUP:]:
    _sync(); t0 = time.perf_counter()
    predictor(a)
    _sync(); times.append(time.perf_counter() - t0)

times = np.array(times) * 1000
timing = {"model": f"maskrcnn_detectron2_{CONFIG}",
          "device": (torch.cuda.get_device_name(0)
                     if torch.cuda.is_available() else "cpu"),
          "n_runs": len(times),
          "mean_ms": float(times.mean()), "std_ms": float(times.std()),
          "median_ms": float(np.median(times)),
          "fps": float(1000.0 / times.mean())}
(OUT_DIR / "timing.json").write_text(json.dumps(timing, indent=2))
print(json.dumps(timing, indent=2))